# Colab template: fetal-shift

Run these cells at the start of **every** Colab session, then run the script for your ticket
(e.g. `python src/train.py --config configs/effb0_s0.yaml`). All real code lives in the repository;
this notebook only prepares the machine.

**One-time setup per account**
1. Add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) holding a GitHub personal
   access token with read access to the repository. Turn on notebook access for it.
2. Akachi only: in Google Drive, open *Shared with me*, right-click `fetal-shift-drive`,
   choose *Organize → Add shortcut* and place it in *My Drive*. The path below then works for both of you.

In [ ]:
# 1. Check the GPU and mount Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Settings: edit REPO_URL once, then leave alone
import os
REPO_URL = 'github.com/<your-username>/fetal-shift.git'   # no https:// prefix
BRANCH     = 'main'
REPO_DIR   = '/content/fetal-shift'
DRIVE_ROOT = '/content/drive/MyDrive/fetal-shift-drive'
LOCAL_ROOT = '/content/fetal-shift-local'
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
os.environ['LOCAL_ROOT'] = LOCAL_ROOT
assert os.path.isdir(DRIVE_ROOT), f'Shared folder not found at {DRIVE_ROOT}; check the Drive shortcut.'

In [ ]:
# 3. Clone the repository, or pull the latest code if it is already here
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} https://{token}@{REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
# 4. Install extra packages (torch is already on Colab)
!pip install -q -r requirements.txt

In [ ]:
# 5. ONE-TIME ONLY (Samuel, first session): create Drive folders, download and extract both datasets.
#    Leave commented out afterwards.
# !python scripts/setup_drive.py --download

In [ ]:
# 6. Copy the data to this machine's local disk (much faster to train from than Drive)
import glob, zipfile, tarfile, shutil
from pathlib import Path
for archive in glob.glob(f'{DRIVE_ROOT}/data/zips/*/*'):
    name = Path(archive).parent.name                       # 'spain' or 'africa'
    dest = Path(LOCAL_ROOT) / 'data/raw' / name
    marker = dest / f'.done_{Path(archive).name}'
    if marker.exists():
        continue
    dest.mkdir(parents=True, exist_ok=True)
    if archive.endswith('.zip'):
        zipfile.ZipFile(archive).extractall(dest)
    elif archive.endswith(('.tar.xz', '.tar.gz', '.tar')):
        tarfile.open(archive).extractall(dest)
    else:
        shutil.copy2(archive, dest)
    marker.touch()
    print('extracted', archive)

# Also unpack the compressed test sets (T3 output), if they exist yet
for archive in glob.glob(f'{DRIVE_ROOT}/data/compressed/*.tar'):
    marker = Path(LOCAL_ROOT) / 'data/compressed' / f'.done_{Path(archive).name}'
    if marker.exists():
        continue
    marker.parent.mkdir(parents=True, exist_ok=True)
    tarfile.open(archive).extractall(LOCAL_ROOT)
    marker.touch()
    print('extracted', archive)

In [ ]:
# 7. Sanity check: count image files per dataset
for name in ['spain', 'africa']:
    root = Path(LOCAL_ROOT) / 'data/raw' / name
    n = sum(1 for p in root.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'})
    print(f'{name}: {n} images in {root}')
# Expected: spain about 12,400 images (all 6 classes); africa 450

# Now run your ticket's script

Examples:
```
!python src/train.py --config configs/effb0_s0.yaml     # Akachi, T4
!python src/infer.py --run effb0_s0                      # Akachi, T5
```
Save everything that must persist to `DRIVE_ROOT` (runs/, outputs/, eval/), never to `/content`:
the local disk is wiped when the session ends.